<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento Digital de Imágenes*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

# 1. Laboratorio 1: Detección Facial Clásica con OpenCV

## 1.1. Objetivo

Construir un pipeline completo de procesamiento clásico (sin aprendizaje profundo) sobre rostros, aplicando los contenidos de las Clases 1 a 4: representación de la imagen como arreglo de NumPy, visualización, espacios de color con OpenCV, segmentación por umbralización en HSV, morfología matemática y detección de bordes. El flujo general es:

$$foto \rightarrow gris \rightarrow ecualizado \rightarrow Haar \rightarrow ROI \rightarrow piel\ HSV \rightarrow morfología \rightarrow ojos/sonrisa + bordes$$

## 1.2. Organización del trabajo

* **Primera hora (práctica):** secciones 2 a 7 de este cuaderno, en orden y sin saltarse celdas. Tiempo total de cómputo inferior a 2 minutos en cualquier portátil.
* **Segunda hora (informe):** completar dentro de este mismo cuaderno las tablas `[E1]`, `[E2]` y las respuestas `[P1]`, `[P2]`, `[P3]`. No se entrega ningún documento adicional.
* **Entrega:** subir a Classroom el archivo `.ipynb` con todas las celdas ejecutadas (con salidas visibles) y las respuestas diligenciadas. La sección 8 verifica que la entrega esté completa antes de subirla.

## 1.3. Requisitos de software

`Python 3.11`, `numpy`, `matplotlib` y `opencv-python 4.x`:

```
pip install "opencv-python<5" numpy matplotlib
```

OpenCV debe ser versión 4.x, pues el detector de Haar (`CascadeClassifier`) fue eliminado en OpenCV 5. La sección 2.1 verifica automáticamente la versión instalada. En Google Colab no necesita instalar nada: el entorno ya trae OpenCV 4.x, `numpy` y `matplotlib`.

## 1.4. Imágenes de trabajo

Este laboratorio trabaja con exactamente **3 fotografías aportadas por el estudiante**, guardadas en la carpeta `img/` junto a este cuaderno, con estos nombres exactos:

| Archivo | Contenido | Propósito |
|---|---|---|
| `frontal.jpg` | Usted de frente, buena iluminación, fondo claro, rostro descubierto (sin gafas de sol ni gorra) | Caso base: todo el pipeline debe funcionar aquí (se espera 1 cara) |
| `grupo_3personas.jpg` | Usted con dos compañeros, los tres de frente a ~2 m, fondo despejado | Caso multiobjeto: conteo y efecto de `scaleFactor` (se esperan 3 caras) |
| `contraluz.jpg` | Un rostro frente a una ventana o fuente de luz (cara más oscura que el fondo) | Caso difícil: efecto de la ecualización y límites de Haar |

Tome las fotos antes de la práctica (celular en JPG, lado mayor ~800 px). Si alguna imagen falta, el cuaderno lo indica en la sección 2.3 y continúa con las disponibles, pero la entrega exige resultados sobre las 3.

**Consentimiento y privacidad:** fotografíe únicamente a personas que acepten participar (incluido usted). Las imágenes se procesan solo en su equipo y el cuaderno resuelto se entrega únicamente por Classroom (uso institucional). No publique las fotos en redes ni las comparta fuera del curso.

## 1.5. Datos del estudiante

Diligencie la siguiente celda con sus datos. Es el primer campo que revisa la calificación.

In [ ]:
ESTUDIANTE_NOMBRE = "Apellidos Nombres"  # <-- reemplazar
ESTUDIANTE_GRUPO = "Grupo"  # <-- reemplazar
FECHA = "2026-09-18"  # <-- reemplazar si es necesario

print(f"Estudiante: {ESTUDIANTE_NOMBRE} | Grupo: {ESTUDIANTE_GRUPO} | Fecha: {FECHA}")

# 2. Preparación del Entorno y de los Datos


## 2.1. Verificación del entorno

Ejecute la celda. Debe imprimir las versiones y confirmar que los tres clasificadores Haar se cargaron (`True`). Si alguna verificación falla, deténgase y revise la instalación indicada en la sección 1.3 antes de continuar.

In [ ]:
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

if not hasattr(cv2, "CascadeClassifier"):
    raise SystemExit(
        f"OpenCV {cv2.__version__} no incluye Haar (eliminado en OpenCV 5). "
        'Instale OpenCV 4.x con: pip install "opencv-python<5" y reinicie el kernel.'
    )

BASE_DIR = Path(__file__).parent if "__file__" in globals() else Path(".")

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_smile.xml")

print("Clasificador rostros cargado:", not face_cascade.empty())
print("Clasificador ojos cargado:", not eye_cascade.empty())
print("Clasificador sonrisa cargado:", not smile_cascade.empty())

# Métricas del laboratorio. Cada ejercicio guarda aquí sus resultados;
# la sección 8 las resume para la entrega y la calificación.
METRICAS = {"nombre": ESTUDIANTE_NOMBRE, "grupo": ESTUDIANTE_GRUPO}

## 2.2. Carga de sus fotografías

Coloque sus 3 fotos (sección 1.4) en la carpeta `img/` con los nombres exactos antes de continuar.
* **En local:** copie los archivos a `img/` junto al cuaderno.
* **En Google Colab:** ejecute la segunda celda de esta sección y seleccione sus 3 fotos cuando el navegador lo pida (deben llamarse `frontal.jpg`, `grupo_3personas.jpg` y `contraluz.jpg`; renómbrelas antes de subirlas).

In [ ]:
import os

EN_COLAB = "COLAB_RELEASE_TAG" in os.environ
IMG_DIR = BASE_DIR / "img"
IMG_DIR.mkdir(exist_ok=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Carpeta img/:", IMG_DIR)

In [ ]:
# Solo en Google Colab: suba sus 3 fotografías. En local esta celda no hace nada.
if EN_COLAB:
    try:
        from google.colab import files

        print("Seleccione sus 3 fotos (frontal.jpg, grupo_3personas.jpg, contraluz.jpg) ...")
        subidas = files.upload()
        for nombre, datos in subidas.items():
            destino = IMG_DIR / Path(nombre).name
            destino.write_bytes(datos)
            print(f"Guardada: {destino} ({len(datos) // 1024} KB)")
    except ImportError:
        print("No se encontró el módulo de subida. Suba las fotos con el panel de archivos de Colab a img/ y vuelva a ejecutar.")
else:
    print("Celda solo para Google Colab. En local, copie sus fotos a img/ (ver sección 1.4).")

## 2.3. Verificación de las imágenes

Ejecute la celda. Para cada una de sus 3 fotos debe aparecer `OK` con sus dimensiones. Si aparece `[FALTA]`, revise el nombre del archivo y su ubicación en `img/` (sección 2.2): puede continuar con las disponibles, pero la entrega requiere las 3.

In [ ]:
IMAGENES = ["frontal.jpg", "grupo_3personas.jpg", "contraluz.jpg"]

imagenes = {}
for nombre in IMAGENES:
    ruta = IMG_DIR / nombre
    img = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if img is None:
        print(f"[FALTA] {nombre}: no está en img/. Coloque su foto con ese nombre (sección 2.2).")
    else:
        print(f"OK {nombre}: {img.shape[1]}x{img.shape[0]} px, dtype={img.dtype}")
        imagenes[nombre] = img

METRICAS["imagenes_encontradas"] = sorted(imagenes.keys())

if not imagenes:
    raise SystemExit("No hay imágenes en img/. No es posible continuar.")

fig, axes = plt.subplots(1, len(imagenes), figsize=(5 * len(imagenes), 5))
if len(imagenes) == 1:
    axes = [axes]
for ax, (nombre, bgr) in zip(axes, imagenes.items()):
    ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(nombre)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2.4. Parámetros del laboratorio

Todos los valores que se experimentan están reunidos aquí. Para los experimentos indicados basta con modificar el parámetro, volver a ejecutar la celda del ejercicio correspondiente y registrar el resultado en la tabla. No es necesario modificar las funciones de la sección 3.

El detector de Haar evalúa ventanas a varias escalas de una pirámide. Dos parámetros controlan el compromiso entre detecciones y falsos positivos:

$$ventanas = \text{escala pirámide}(scaleFactor) \times \text{votos mínimos}(minNeighbors)$$

* `SCALE_FACTOR` cercano a 1.0 recorre más escalas: detecta más, pero tarda más.
* `MIN_NEIGHBORS` alto exige más acuerdo entre ventanas vecinas: menos falsos positivos, pero pierde caras difíciles.

In [ ]:
# --- Preproceso ---
GAUSS_KSIZE = (5, 5)  # Tamaño del kernel gaussiano. Debe ser impar.
SIGMA = 0  # 0 = desviación calculada automáticamente desde el tamaño.

# --- Detección Haar (Ejercicio 2: experimentar con estos dos) ---
SCALE_FACTOR = 1.1  # Experimento: comparar 1.05 frente a 1.3
MIN_NEIGHBORS = 5  # Experimento opcional: comparar 3 frente a 8
MIN_SIZE = (60, 60)  # Se ignoran detecciones menores (ruido).

# --- Segmentación de piel en HSV (Ejercicio 3) ---
# Un píxel se clasifica como piel si sus tres canales están dentro del rango:
# $$M(x,y) = 1 \iff H_{low} \le H \le H_{high} \land S_{low} \le S \le S_{high} \land V_{low} \le V \le V_{high}$$
# Rangos de OpenCV: H en 0-179, S y V en 0-255.
PIEL_LOW = np.array([0, 30, 60], dtype=np.uint8)
PIEL_HIGH = np.array([20, 150, 255], dtype=np.uint8)

# Morfología: la apertura elimina puntos blancos aislados (sal) y el cierre
# rellena huecos negros (pimienta):
# $$A \circ B = (A \ominus B) \oplus B \quad ; \quad A \bullet B = (A \oplus B) \ominus B$$
KERNEL_OPEN = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))  # Experimento: comparar (3,3) frente a (7,7)
KERNEL_CLOSE = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# --- Bordes (Ejercicio 4) ---
CANNY_T1 = 100  # Umbral bajo de histéresis.
CANNY_T2 = 200  # Umbral alto de histéresis. Experimento opcional: (50,150) frente a (100,200).

FIGSIZE = (15, 8)
SALIDA_PNG = BASE_DIR / "resultado.png"

# 3. Funciones de Procesamiento

Estas funciones implementan el pipeline. Léalas para asociar cada operación con su fundamento (ecualización por acumulada $s_k = (L-1)\sum p(r_j)$, suavizado gaussiano $G(x,y)$, voto de Haar, `inRange`, apertura/cierre). No requieren modificaciones.

In [ ]:
def preprocesar(bgr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Convierte BGR a gris, suaviza y ecualiza. Devuelve (gris, gris_ecualizado)."""
    gris = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gris = cv2.GaussianBlur(gris, GAUSS_KSIZE, SIGMA)
    gris_eq = cv2.equalizeHist(gris)
    return gris, gris_eq


def detectar_rostros(gris: np.ndarray) -> np.ndarray:
    """Aplica Haar sobre una imagen en gris. Devuelve arreglos (x, y, w, h)."""
    t0 = time.time()
    rostros = face_cascade.detectMultiScale(
        gris,
        scaleFactor=SCALE_FACTOR,
        minNeighbors=MIN_NEIGHBORS,
        minSize=MIN_SIZE,
    )
    dt = time.time() - t0
    print(f"  Haar: {len(rostros)} cara(s) en {dt:.2f}s (scale={SCALE_FACTOR}, neighbors={MIN_NEIGHBORS})")
    return rostros


def segmentar_piel_hsv(bgr_roi: np.ndarray) -> np.ndarray:
    """Máscara binaria de piel en HSV mediante inRange."""
    hsv = cv2.cvtColor(bgr_roi, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, PIEL_LOW, PIEL_HIGH)


def limpiar_mascara(mask: np.ndarray) -> np.ndarray:
    """Apertura para quitar sal y cierre para tapar pimienta."""
    abierta = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL_OPEN)
    return cv2.morphologyEx(abierta, cv2.MORPH_CLOSE, KERNEL_CLOSE)


def dibujar(bgr: np.ndarray, rostros=(), ojos=(), sonrisas=()) -> np.ndarray:
    """Devuelve una copia anotada: verde = rostro, azul = ojos, magenta = sonrisa."""
    out = bgr.copy()
    for (x, y, w, h) in rostros:
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(out, "cara", (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
    for (x, y, w, h) in ojos:
        cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 0), 2)
    for (x, y, w, h) in sonrisas:
        cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 255), 2)
    return out

# 4. Ejercicio 1 — Preproceso y Ecualización [10 min]

**Procedimiento:**
1. Ejecute la celda y observe las tres columnas: imagen original, gris con suavizado y gris ecualizado, con sus histogramas debajo.
2. Compare el histograma del gris frente al ecualizado: la ecualización redistribuye las intensidades según la acumulada $s_k = (L-1)\sum_{j=0}^{k} p(r_j)$ con $p(r_j)=n_j/MN$.
3. Este ejercicio es formativo (sin entregable): su propósito es visualizar por qué la ecualización ayuda en `contraluz.jpg` antes de detectar.

In [ ]:
demo_nombre = "contraluz.jpg" if "contraluz.jpg" in imagenes else next(iter(imagenes))
demo = imagenes[demo_nombre]
gris_demo, gris_eq_demo = preprocesar(demo)

fig, axes = plt.subplots(2, 3, figsize=FIGSIZE)
axes[0, 0].imshow(cv2.cvtColor(demo, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f"Original ({demo_nombre})")
axes[0, 0].axis("off")
axes[0, 1].imshow(gris_demo, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Gris + blur")
axes[0, 1].axis("off")
axes[0, 2].imshow(gris_eq_demo, cmap="gray", vmin=0, vmax=255)
axes[0, 2].set_title("Gris ecualizado")
axes[0, 2].axis("off")
axes[1, 0].hist(demo.ravel(), bins=128, range=[0, 256], color="gray")
axes[1, 0].set_title("Histograma BGR")
axes[1, 1].hist(gris_demo.ravel(), bins=256, range=[0, 256], color="black")
axes[1, 1].set_title("Histograma gris")
axes[1, 2].hist(gris_eq_demo.ravel(), bins=256, range=[0, 256], color="red")
axes[1, 2].set_title("Histograma ecualizado")
plt.tight_layout()
plt.show()

# 5. Ejercicio 2 — Detección de Rostros con Haar [20 min]

**Procedimiento:**
1. Ejecute la celda con los valores por defecto (`SCALE_FACTOR = 1.1`). Registre en la Tabla `[E1]` el número de caras sobre gris y sobre gris ecualizado para cada una de las 3 imágenes. Si el conteo supera el número de personas visibles, no es un error: son falsos positivos; regístrelos tal cual y explíquelos en `[P1]`.
2. Experimento obligatorio: cambie en la sección 2.4 `SCALE_FACTOR = 1.3`, vuelva a ejecutar la celda y agregue la fila correspondiente a `grupo_3personas.jpg`. Restaure después el valor 1.1.
3. Experimento opcional: compare `MIN_NEIGHBORS = 3` frente a `8` en `contraluz.jpg`.
4. Responda `[P1]` debajo de la tabla.

In [ ]:
resultados = {}

for nombre, bgr in imagenes.items():
    print(f"--- {nombre} ---")
    gris, gris_eq = preprocesar(bgr)
    rostros_gris = detectar_rostros(gris)
    rostros_eq = detectar_rostros(gris_eq)
    print(f"  gris={len(rostros_gris)} frente a ecualizado={len(rostros_eq)} (conserve el ecualizado)")
    resultados[nombre] = {"gris": len(rostros_gris), "eq": len(rostros_eq), "rostros": rostros_eq}
    anotada = dibujar(bgr, rostros_eq)
    plt.figure(figsize=(6, 5))
    plt.imshow(cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB))
    plt.title(f"{nombre}: {len(rostros_eq)} cara(s) [scale={SCALE_FACTOR}]")
    plt.axis("off")
    plt.show()

METRICAS["deteccion"] = {k: {"gris": v["gris"], "ecualizado": v["eq"]} for k, v in resultados.items()}
METRICAS["scale_factor"] = SCALE_FACTOR
METRICAS["min_neighbors"] = MIN_NEIGHBORS

print("gris | ecualizado (para transcribir a [E1])")
for nombre, r in resultados.items():
    print(f"{nombre} | {r['gris']} | {r['eq']}")

## Tabla [E1] — Resultados de detección (calificable)

Transcriba aquí los conteos impresos por la celda anterior. La fila de `grupo_3personas.jpg` con `scale=1.3` corresponde al experimento obligatorio del paso 2.

| imagen | gris #caras | ecualizado #caras | scale | tiempo aprox. | observación (1 línea) |
|---|---|---|---|---|---|
| frontal.jpg |  |  | 1.1 |  |  |
| grupo_3personas.jpg |  |  | 1.1 |  |  |
| grupo_3personas.jpg |  |  | 1.3 |  |  |
| contraluz.jpg |  |  | 1.1 |  |  |

*(Diligencie todas las celdas de la tabla. Sin esta tabla la entrega está incompleta.)*

## Pregunta [P1] — Efecto de `scaleFactor` (calificable)

Con los datos de la Tabla [E1]: ¿qué configuración detectó más caras en el grupo y cuál fue más rápida? Explique con la pirámide de escalas por qué un `scaleFactor` pequeño tarda más.

*(Escriba su respuesta aquí. Mínimo 4 líneas con argumento técnico. No deje el marcador.)*

# 6. Ejercicio 3 — Piel en HSV y Limpieza Morfológica [15 min]

**Procedimiento:**
1. Ejecute la celda: toma el rostro más grande de `frontal.jpg` (o de la primera imagen con detección), segmenta piel en HSV y compara la máscara cruda frente a la limpia con `OPEN 3x3 + CLOSE 5x5`.
2. Experimento obligatorio: cambie en la sección 2.4 `KERNEL_OPEN` a `(7,7)`, vuelva a ejecutar y registre ambos conteos de píxeles en la Tabla `[E2]`. Restaure `(3,3)`.
3. Si Haar no detectó ningún rostro, la celda usa un recorte central para no interrumpir el laboratorio: indíquelo en `[E2]` como fallo de Haar.
4. Responda `[P2]`.

In [ ]:
bgr_ref_nombre = (
    "frontal.jpg"
    if "frontal.jpg" in resultados and len(resultados["frontal.jpg"]["rostros"]) > 0
    else next((k for k, v in resultados.items() if len(v["rostros"]) > 0), None)
)
roi_desde_deteccion = True

if bgr_ref_nombre is not None:
    bgr_ref = imagenes[bgr_ref_nombre]
    rostros_ref = resultados[bgr_ref_nombre]["rostros"]
    x, y, w, h = max(rostros_ref, key=lambda r: r[2] * r[3])
    print(f"ROI de {bgr_ref_nombre}: x={x} y={y} w={w} h={h}")
else:
    roi_desde_deteccion = False
    bgr_ref_nombre = next(iter(imagenes))
    bgr_ref = imagenes[bgr_ref_nombre]
    H, W = bgr_ref.shape[:2]
    x, y, w, h = W // 4, H // 4, W // 2, H // 2
    print(f"Haar no detectó rostros. Se usa un recorte central de {bgr_ref_nombre}.")

roi = bgr_ref[y : y + h, x : x + w]
mask_cruda = segmentar_piel_hsv(roi)
mask_limpia = limpiar_mascara(mask_cruda)
n_cruda, n_limpia = int(np.count_nonzero(mask_cruda)), int(np.count_nonzero(mask_limpia))
print(f"Piel: cruda={n_cruda} px, limpia={n_limpia} px")

METRICAS["piel"] = {
    "imagen": bgr_ref_nombre,
    "roi_desde_deteccion": roi_desde_deteccion,
    "cruda_px": n_cruda,
    "limpia_px": n_limpia,
    "kernel_open": KERNEL_OPEN.shape[:2],
}

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE)
axes[0].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
axes[0].set_title("ROI rostro")
axes[0].axis("off")
axes[1].imshow(mask_cruda, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Máscara HSV cruda")
axes[1].axis("off")
axes[2].imshow(mask_limpia, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("OPEN + CLOSE")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## Tabla [E2] — Máscara de piel (calificable)

| kernel OPEN | máscara cruda (px) | máscara limpia (px) | ¿ROI desde detección o recorte central? |
|---|---|---|---|
| (3,3) |  |  |  |
| (7,7) |  |  |  |

*(Transcriba los conteos impresos por la celda para cada kernel.)*

## Pregunta [P2] — Apertura y cierre (calificable)

Compare ambas filas de la Tabla [E2]: ¿qué eliminó la apertura (ruido sal) y qué rellenó el cierre (pimienta)? ¿Qué ocurrió con el kernel de (7,7): limpió más o destruyó piel válida?

*(Escriba su respuesta aquí. Mínimo 4 líneas con argumento técnico. No deje el marcador.)*

# 7. Ejercicio 4 — Rasgos, Bordes y Figura Final [15 min]

**Procedimiento:**
1. Ejecute la primera celda: detecta ojos y sonrisa únicamente dentro del ROI del rostro (por eso es rápido en local) y compara el detector de bordes de Canny frente al gradiente morfológico.
2. El detector de Canny es multietapa (suavizado gaussiano → gradientes de Sobel → supresión de no-máximos → histéresis con `CANNY_T1/T2`):

$$M = \sqrt{G_x^2+G_y^2}, \quad fuerte = M>T_2, \; débil = T_1<M\le T_2$$

El gradiente morfológico no usa derivadas, solo geometría:

$$G = (A \oplus B)-(A \ominus B)$$

3. Ejecute la segunda celda: genera el panel 2x3 y lo guarda como `resultado.png`. Esa imagen es el entregable `[FIG]`.
4. Responda `[P3]`.

In [ ]:
gris_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
ojos = eye_cascade.detectMultiScale(gris_roi, scaleFactor=1.1, minNeighbors=8, minSize=(20, 20))
sonrisas = smile_cascade.detectMultiScale(gris_roi, scaleFactor=1.3, minNeighbors=25, minSize=(30, 30))
print(f"Ojos: {len(ojos)}, sonrisas: {len(sonrisas)} (puede fallar con gafas o perfil: regístrelo en [P3])")

canny = cv2.Canny(gris_roi, CANNY_T1, CANNY_T2)
kernel_g = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
grad_morf = cv2.morphologyEx(gris_roi, cv2.MORPH_GRADIENT, kernel_g)
anotada_roi = dibujar(roi, [], ojos, sonrisas)

METRICAS["rasgos"] = {"ojos": len(ojos), "sonrisas": len(sonrisas)}
METRICAS["canny"] = [CANNY_T1, CANNY_T2]

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE)
axes[0].imshow(cv2.cvtColor(anotada_roi, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Ojos {len(ojos)} / sonrisas {len(sonrisas)}")
axes[0].axis("off")
axes[1].imshow(canny, cmap="gray")
axes[1].set_title(f"Canny ({CANNY_T1},{CANNY_T2})")
axes[1].axis("off")
axes[2].imshow(grad_morf, cmap="gray")
axes[2].set_title("Gradiente morfológico")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
gris_full = cv2.cvtColor(bgr_ref, cv2.COLOR_BGR2GRAY)
canny_full = cv2.Canny(gris_full, CANNY_T1, CANNY_T2)
n_caras_panel = len(resultados[bgr_ref_nombre]["rostros"]) if bgr_ref_nombre in resultados else 0
anotada_full = dibujar(bgr_ref, resultados[bgr_ref_nombre]["rostros"] if bgr_ref_nombre in resultados else [])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(cv2.cvtColor(bgr_ref, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("1. Original")
axes[0, 0].axis("off")
axes[0, 1].imshow(gris_full, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("2. Gris")
axes[0, 1].axis("off")
axes[0, 2].imshow(cv2.cvtColor(anotada_full, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title(f"3. Haar: {n_caras_panel} cara(s)")
axes[0, 2].axis("off")
axes[1, 0].imshow(mask_limpia, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("4. Piel HSV limpia")
axes[1, 0].axis("off")
axes[1, 1].imshow(cv2.cvtColor(anotada_roi, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("5. Ojos y sonrisa")
axes[1, 1].axis("off")
axes[1, 2].imshow(canny_full, cmap="gray")
axes[1, 2].set_title("6. Canny")
axes[1, 2].axis("off")
plt.tight_layout()
plt.savefig(SALIDA_PNG, dpi=150)
print(f"Figura [FIG] guardada en: {SALIDA_PNG}")
plt.show()

## Pregunta [P3] — Límites del enfoque clásico (calificable)

¿En qué imagen o condición falló Haar (perfil, contraluz, oclusión, grupo)? ¿Por qué el espacio HSV separa la piel mejor que BGR? Con lo observado, ¿por qué un detector aprendido (Unidad 2) superaría a este pipeline?

*(Escriba su respuesta aquí. Mínimo 4 líneas con argumento técnico. No deje el marcador.)*

# 8. Entrega en Classroom

**Procedimiento:**
1. Ejecute la celda de verificación. Debe mostrar `ENTREGA LISTA`. Si indica faltantes, devuélvase a la sección señalada.
2. En el menú del cuaderno seleccione *Reiniciar kernel y ejecutar todo*, espere a que termine sin errores y guarde.
3. Suba a Classroom el archivo `.ipynb` de este cuaderno (con salidas visibles). No suba el `.py` ni las imágenes. En Google Colab, descárguelo primero con *Archivo > Descargar > Descargar .ipynb*.

La calificación (humana o automática) revisa los identificadores `[E1]`, `[P1]`, `[E2]`, `[P2]`, `[P3]` y `[FIG]` dentro de este cuaderno, con los siguientes pesos:

| ID | Contenido | Peso |
|---|---|---|
| Datos | `ESTUDIANTE_NOMBRE` y `GRUPO` diligenciados (sección 1.5) | 5% |
| `[E1]` | Tabla de detección completa: sus 3 fotos + fila `scale=1.3` en el grupo | 25% |
| `[P1]` | Efecto de `scaleFactor`, explicado con la pirámide de escalas | 15% |
| `[E2]` | Tabla de piel con ambos kernels | 15% |
| `[P2]` | Apertura frente a cierre + efecto del kernel (7,7) | 15% |
| `[P3]` | Fallos de Haar + HSV frente a BGR + puente a detectores aprendidos | 15% |
| `[FIG]` | `resultado.png` generado y cuaderno ejecutado sin errores | 10% |

Los conteos de `[E1]` deben coincidir con las salidas impresas por el cuaderno (cada estudiante tiene fotos distintas: se califica la coherencia, no un número fijo). Las respuestas `[P1]`–`[P3]` exigen argumento técnico; un marcador sin reemplazar vale cero.

In [ ]:
import json

pendientes = []
if ESTUDIANTE_NOMBRE.strip().lower() in ("", "apellidos nombres"):
    pendientes.append("datos del estudiante (sección 1.5)")
if len(METRICAS.get("imagenes_encontradas", [])) < 3:
    pendientes.append(f"imágenes: se encontraron {METRICAS.get('imagenes_encontradas')}")
if "deteccion" not in METRICAS:
    pendientes.append("Ejercicio 2 sin ejecutar")
if "piel" not in METRICAS:
    pendientes.append("Ejercicio 3 sin ejecutar")
if "rasgos" not in METRICAS:
    pendientes.append("Ejercicio 4 sin ejecutar")
if not SALIDA_PNG.exists():
    pendientes.append("figura [FIG] (resultado.png no generado)")

print("=== RESUMEN DE ENTREGA (legible por máquina) ===")
print(json.dumps(METRICAS, indent=1, default=str))
print()
if pendientes:
    print("FALTA POR COMPLETAR:")
    for p in pendientes:
        print(f"  - {p}")
    print("Verifique además que las tablas [E1], [E2] y las respuestas [P1], [P2], [P3] estén diligenciadas.")
else:
    print("ENTREGA LISTA: verifique que [E1], [E2], [P1], [P2] y [P3] estén diligenciadas y suba el .ipynb a Classroom.")